In [15]:
from pathlib import Path
from typing import Optional, Union
import uuid
import numpy as np
import pyvista as pv
import os
import tempfile
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

try:
    from IPython import get_ipython

    if get_ipython():
        # Running inside a Jupyter notebook
        matplotlib.use("module://matplotlib_inline.backend_inline", force=True)
        notebook_mode = True
    else:
        # Running in a normal desktop Python process
        matplotlib.use("Qt5Agg", force=True)
        notebook_mode = False
except ImportError:
    # Fallback for headless servers (no display)
    matplotlib.use("Agg", force=True)
    notebook_mode = False


def _ensure_output_path(output_path: Optional[Path]) -> Path:
    """Create parent directories and return an absolute output path."""
    if output_path is None:
        output_path = (
            Path(tempfile.gettempdir()) / f"imagery_all_stateful_{uuid.uuid4().hex}.png"
        )

    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


def _rotation_matrix(axis, angle_rad):
    axis = axis / np.linalg.norm(axis)
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    t = 1 - c
    x, y, z = axis
    return np.array(
        [
            [t * x * x + c, t * x * y - s * z, t * x * z + s * y],
            [t * x * y + s * z, t * y * y + c, t * y * z - s * x],
            [t * x * z - s * y, t * y * z + s * x, t * z * z + c],
        ]
    )


class StatefulImageryWithInitialRotationModule_8:
    def __init__(self, bounds_map: dict, off_screen=True, show_grid=True):
        self.bounds_map = bounds_map
        self.off_screen = off_screen
        self.seq_count_map = {target: 0 for target in self.bounds_map.keys()}
        self.show_grid = show_grid
        self.plotter_map = self._build()
        self._initial_rotation()
        self.count = 0

    def _initial_rotation(self):
        # apply initial rotation
        for target, bounds_data in self.bounds_map.items():
            if bounds_data["rotation"]:
                self.run_human_sequence_and_save_image(
                    target, bounds_data["rotation"], no_save=True
                )

    def _build(self):
        p_map = {}
        # Ensure we can run headless on linux servers if needed
        # pv.start_xvfb() # Uncomment if running on a headless Linux server without X11

        for target, bounds_data in self.bounds_map.items():
            _plotter = pv.Plotter(window_size=(800, 600), off_screen=self.off_screen)
            _plotter.set_background("white")

            for b in bounds_data["bounds"]:
                _plotter.add_mesh(
                    pv.Cube(bounds=b),
                    color="#B8C0CC",      # lighter fill color
                    # edge_color="#111111", # darker edge/aresta color
                    # line_width=3,         # thicker edges
                    show_edges=True,
                    lighting=True,  # enable shading
                    ambient=0.55,  # raise ambient -> less harsh shadows
                    diffuse=0.45,  # lower diffuse -> less contrast
                    specular=0.05,  # reduce shiny highlights
                    specular_power=0.05,  # broader/softer highlight
                )

            _plotter.reset_camera()
            _plotter.camera_position = "iso"

            if self.show_grid:
                _plotter.show_grid()

            p_map[target] = _plotter

        return p_map

    def close(self):
        for plotter in self.plotter_map.values():
            try:
                plotter.close()
                plotter.deep_clean()
            except Exception:
                pass
        self.plotter_map.clear()
        pv.close_all()

    def rotate(self, target: str, mode="yaw", value: Union[float, str] = 30.0):
        """
        Rotates the camera or resets the view.

        Args:
            target: The key in bounds_map.
            mode: 'yaw', 'pitch', 'roll', or 'reset'.
            value:
                - If mode is rotation: degrees (float).
                - If mode is reset: axis string ('x', 'y', 'z').
        """
        cam = self.plotter_map[target].camera
        pos = np.array(cam.position)
        focal = np.array(cam.focal_point)
        up = np.array(cam.up)

        # Calculate current view vector and distance
        view = focal - pos
        distance = np.linalg.norm(view)
        view /= distance  # normalize

        if mode == "reset":
            if value == "iso":
                # Place the camera along the (1, 1, 1) direction at the current distance
                iso_dir = np.array([1.0, 1.0, 1.0])
                iso_dir /= np.linalg.norm(iso_dir)

                new_pos = focal + iso_dir * distance
                cam.position = tuple(new_pos)
                cam.up = (0.0, 0.0, 1.0)
            else:
                # Handle Axis Reset
                axis_char = str(value).lower()

                # We maintain the current focal point and distance (zoom level),
                # but move the position to align with the requested axis.
                if axis_char == "x":
                    # View from +X looking towards focal point
                    new_pos = focal + np.array([distance, 0, 0])
                    new_up = np.array([0, 0, 1])  # Z is up
                elif axis_char == "y":
                    # View from +Y looking towards focal point
                    new_pos = focal + np.array([0, distance, 0])
                    new_up = np.array([0, 0, 1])  # Z is up
                elif axis_char == "z":
                    # View from +Z looking down (Top view)
                    new_pos = focal + np.array([0, 0, distance])
                    new_up = np.array([0, 1, 0])  # Y is up usually for maps/top-down
                else:
                    print(f"Warning: Unknown reset axis '{axis_char}', ignoring.")
                    return

                cam.position = tuple(new_pos)
                cam.up = tuple(new_up)

        else:
            # Handle Rotations
            step = np.deg2rad(float(value))

            if mode == "yaw":
                # Rotate around global Z
                axis = np.array([0.0, 0.0, 1.0])
            elif mode == "pitch":
                # Rotate around the camera’s right vector
                right = np.cross(view, up)
                if np.linalg.norm(right) < 1e-6:
                    # Fallback if looking straight down/up
                    right = np.array([1.0, 0.0, 0.0])
                else:
                    right /= np.linalg.norm(right)
                axis = right
            elif mode == "roll":
                # Spin around view axis
                axis = view
            else:
                raise ValueError(
                    f"mode must be 'yaw', 'pitch', 'roll', or 'reset'. received {mode}"
                )

            R = _rotation_matrix(axis, step)
            cam.position = tuple(focal + R @ (pos - focal))
            cam.up = tuple(R @ up)

        # Force render update
        self.plotter_map[target].render()

    def show(self, target):
        self.plotter_map[target].show()

    def run_sequence_and_save_image(
        self, target, action_sequence: str, output_path: str = None
    ):
        """
        Execute a sequence of rotations/resets.
        Example: "reset:x, yaw:45, pitch:30, reset:z"
        """

        # Calculate azimuth/elevation from position
        def get_spherical_coords(cam):
            pos = np.array(cam.position) - np.array(cam.focal_point)
            r = np.linalg.norm(pos)
            el = np.degrees(np.arcsin(pos[2] / r))
            az = np.degrees(np.arctan2(pos[1], pos[0]))
            return az, el

        # Prepare output dir
        rand_part = f"{random.randint(1000,9999)}_{random.getrandbits(32)}"
        temp_dir = tempfile.mkdtemp(prefix=f"stateful_imagery_steps_{rand_part}_")
        steps = []
        cmds = [cmd.strip() for cmd in action_sequence.split(",") if cmd.strip()]

        for i, cmd in enumerate(cmds):
            if ":" not in cmd:
                raise ValueError(f"Invalid command fragment '{cmd}' in action sequence")

            mode, val_str = cmd.split(":", 1)
            mode = mode.strip().lower()
            val_str = val_str.strip()

            # Parse value: float for rotations, string for reset
            if mode == "reset":
                val = val_str  # keep as string 'x', 'y', 'z'
                display_val = val_str
            else:
                try:
                    val = float(val_str)
                    display_val = f"{int(val)}"  # formatted for label
                except ValueError:
                    raise ValueError(
                        f"Value for {mode} must be a number, got '{val_str}'"
                    )

            # Execute
            self.rotate(target, mode, val)

            # Save step image
            fname = os.path.join(
                temp_dir, f"step_{i:03d}_{mode}_{str(val).replace('.','_')}.png"
            )
            self.plotter_map[target].screenshot(fname)

            # Get camera params for labeling
            cam = self.plotter_map[target].camera
            command_str = f"{mode}:{display_val}"

            # Safely get azimuth/elevation if available (depends on vtk version/backend)
            # az = getattr(cam, 'azimuth', 0)
            # el = getattr(cam, 'elevation', 0)
            az, el = get_spherical_coords(cam)

            label = (
                f"seq:{self.seq_count_map[target]} [{command_str}]"
                # f"pos:[{cam.position[0]:.1f},{cam.position[1]:.1f},{cam.position[2]:.1f}]\n"
                # f"az:{az:.1f}° el:{el:.1f}°"
            )
            steps.append((fname, label))

        # Compose Grid Image
        n = len(steps)
        cols = 4
        rows = (n + cols - 1) // cols

        # Calculate figure size (heuristic)
        fig = plt.figure(figsize=(16, 4 * rows))

        for i, (fname, label) in enumerate(steps, 1):
            img = mpimg.imread(fname)
            ax = plt.subplot(rows, cols, i)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(label, fontsize=15, color="black", wrap=True)

        plt.subplots_adjust(wspace=0.1, hspace=0.3)
        plt.tight_layout()
        st = plt.suptitle(
            f"Target: {target}",
            fontsize=18,
            color="black",
            y=1.00 + (0.01 * rows),
            ha="left",
        )
        st.set_x(0.0)

        composed_path = _ensure_output_path(output_path)
        plt.savefig(composed_path, bbox_inches="tight", pad_inches=0.1)
        plt.close(fig)

        # Show if in notebook, safely ignore if background process
        if notebook_mode:
            try:
                # Re-open image to display in notebook output
                img_final = mpimg.imread(composed_path)
                plt.figure(figsize=(12, 12))
                plt.imshow(img_final)
                plt.axis("off")
                plt.show()
            except Exception:
                pass

        # Cleanup
        try:
            import shutil

            shutil.rmtree(temp_dir)
        except Exception as e:
            print(f"Warning: failed to clean up temp_dir {temp_dir}: {e}")

        print(f"Composed summary image written to {composed_path}")
        return composed_path

    def rotate_human(self, target: str, action: str, value=None):
        """
        Human-friendly camera commands with intuitive behavior from any angle.

        Supported:
          - left:<deg>   -> orbit left (camera-relative)
          - right:<deg>  -> orbit right (camera-relative)
          - up:<deg>     -> orbit up (camera-relative)
          - down:<deg>   -> orbit down (camera-relative)
          - rotate:cw:<deg>  -> roll clockwise
          - rotate:ccw:<deg> -> roll counter-clockwise
          - view:x|y|z|iso -> reset to axis view

        Notes:
          - left/right/up/down always orbit around the object intuitively
          - These work correctly even from top-down or bottom-up views
        """
        self.seq_count_map[target] += 1
        action = str(action).strip().lower()
        cam = self.plotter_map[target].camera

        # View command (unchanged)
        if action == "view":
            if value is None:
                raise ValueError("view requires a value: x, y, z, or iso")
            return self.rotate(target, mode="reset", value=str(value).strip().lower())

        # Roll command (unchanged - it's already correct)
        if action == "rotate":
            if value is None:
                raise ValueError(
                    "rotate requires a direction and degrees: rotate:cw:30 or rotate:ccw:30"
                )

            if isinstance(value, (tuple, list)) and len(value) == 2:
                direction = str(value[0]).strip().lower()
                deg = float(value[1])
            else:
                parts = str(value).strip().lower().split(":")
                if len(parts) != 2:
                    raise ValueError("rotate value must be 'cw:<deg>' or 'ccw:<deg>'")
                direction, deg_str = parts
                deg = float(deg_str)

            if direction not in {"cw", "ccw"}:
                raise ValueError("rotate direction must be 'cw' or 'ccw'")

            signed = -deg if direction == "cw" else +deg
            return self.rotate(target, mode="roll", value=signed)

        # Human-friendly orbit controls (NEW IMPLEMENTATION)
        if action in {"left", "right", "up", "down"}:
            if value is None:
                raise ValueError(f"{action} requires degrees, e.g. '{action}:30'")
            deg = float(value)

            # Get current camera vectors
            pos = np.array(cam.position)
            focal = np.array(cam.focal_point)
            up = np.array(cam.up)

            # Calculate view direction
            view = focal - pos
            distance = np.linalg.norm(view)
            view /= distance

            # Calculate camera's right vector (perpendicular to view and up)
            right = np.cross(view, up)
            right_norm = np.linalg.norm(right)

            # Handle degenerate case (looking straight up/down)
            if right_norm < 1e-6:
                # When looking along up vector, derive right from world coordinates
                # Assume world up is [0, 0, 1]
                world_up = np.array([0.0, 0.0, 1.0])
                # Use a perpendicular vector in the XY plane
                if abs(view[2]) > 0.99:  # Nearly vertical view
                    right = (
                        np.array([1.0, 0.0, 0.0])
                        if view[2] > 0
                        else np.array([-1.0, 0.0, 0.0])
                    )
                else:
                    right = np.cross(view, world_up)
                    right /= np.linalg.norm(right)
            else:
                right /= right_norm

            # Recalculate proper up vector (perpendicular to both view and right)
            up_corrected = np.cross(right, view)
            up_corrected /= np.linalg.norm(up_corrected)

            # Choose rotation axis based on action
            if action == "left":
                # Rotate around the "up" axis (orbit left)
                axis = up_corrected
                angle = np.deg2rad(deg)
            elif action == "right":
                # Rotate around the "up" axis (orbit right)
                axis = up_corrected
                angle = np.deg2rad(-deg)
            elif action == "up":
                # Rotate around the "right" axis (orbit up)
                axis = right
                angle = np.deg2rad(deg)
            elif action == "down":
                # Rotate around the "right" axis (orbit down)
                axis = right
                angle = np.deg2rad(-deg)

            # Apply rotation
            R = _rotation_matrix(axis, angle)

            # Rotate camera position around focal point
            new_pos = focal + R @ (pos - focal)
            cam.position = tuple(new_pos)

            # Rotate up vector to maintain orientation
            new_up = R @ up
            cam.up = tuple(new_up)

            # Force render update
            self.plotter_map[target].render()
            return

        raise ValueError(
            f"Unknown human action '{action}'. Valid: left/right/up/down/rotate/view"
        )

    def run_human_sequence_and_save_image(
        self,
        target,
        action_sequence: str,
        output_path: str = None,
        no_save: bool = False,
        image_title: str = None,
    ):
        """
        Execute a sequence of human-friendly commands and save a composed summary image.

        Example:
          "view:y, up:30, up:30, left:45, rotate:cw:20"

        This reuses the same screenshot+grid composition logic as run_sequence_and_save_image,
        but parses commands in the human format and internally calls rotate_human().
        """

        # ---- same helper as your existing method ----
        def get_spherical_coords(cam):
            pos = np.array(cam.position) - np.array(cam.focal_point)
            r = np.linalg.norm(pos)
            el = np.degrees(np.arcsin(pos[2] / r))
            az = np.degrees(np.arctan2(pos[1], pos[0]))
            return az, el

        rand_part = f"{random.randint(1000,9999)}_{random.getrandbits(32)}"
        temp_dir = tempfile.mkdtemp(prefix=f"stateful_imagery_human_steps_{rand_part}_")
        steps = []
        cmds = [cmd.strip() for cmd in action_sequence.split(",") if cmd.strip()]

        for i, cmd in enumerate(cmds):
            # Allowed forms:
            #   left:30
            #   rotate:cw:30
            #   view:y
            parts = [p.strip() for p in cmd.split(":")]

            if len(parts) < 2:
                raise ValueError(
                    f"Invalid command fragment '{cmd}' (needs at least one ':')"
                )

            action = parts[0].lower()

            if action == "rotate":
                if len(parts) != 3:
                    raise ValueError(
                        f"Invalid rotate command '{cmd}'. Use rotate:cw:<deg> or rotate:ccw:<deg>"
                    )
                direction = parts[1].lower()
                deg = float(parts[2])
                human_value = f"{direction}:{deg}"
                self.rotate_human(target, "rotate", f"{direction}:{deg}")
                label_cmd = (
                    f"rotate:{direction}:{int(deg) if deg.is_integer() else deg}"
                )

            elif action == "view":
                if len(parts) != 2:
                    raise ValueError(
                        f"Invalid view command '{cmd}'. Use view:x|y|z|iso"
                    )
                axis = parts[1].lower()
                human_value = axis
                self.rotate_human(target, "view", axis)
                label_cmd = f"view:{axis}"

            else:
                # left/right/up/down:<deg>
                if len(parts) != 2:
                    raise ValueError(f"Invalid command '{cmd}'. Use {action}:<deg>")
                deg = float(parts[1])
                human_value = deg
                self.rotate_human(target, action, deg)
                label_cmd = f"{action}:{int(deg) if deg.is_integer() else deg}"

            # Save step image
            fname = os.path.join(
                temp_dir,
                f"step_{i:03d}_{action}_{str(human_value).replace('.','_').replace(':','-')}.png",
            )
            self.plotter_map[target].screenshot(fname)

            cam = self.plotter_map[target].camera
            az, el = get_spherical_coords(cam)
            label = (
                f"seq:{self.seq_count_map[target]} [{label_cmd}]"
                # f"pos:[{cam.position[0]:.1f},{cam.position[1]:.1f},{cam.position[2]:.1f}]\n"
                # f"az:{az:.1f}° el:{el:.1f}°"
            )
            steps.append((fname, label))

        # Compose grid (same as your existing method)
        if no_save:  # skip save
            return None

        n = len(steps)
        cols = 4
        rows = (n + cols - 1) // cols
        fig = plt.figure(figsize=(16, 4 * rows))

        for i, (fname, label) in enumerate(steps, 1):
            img = mpimg.imread(fname)
            ax = plt.subplot(rows, cols, i)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(label, fontsize=15, color="black", wrap=True)

        plt.subplots_adjust(wspace=0.1, hspace=0.3)
        plt.tight_layout()
        _title = image_title if image_title else f"Image {label}: {target}"
        st = plt.suptitle(
            _title,
            fontsize=18,
            color="black",
            y=1.00 + (0.01 * rows),
            ha="left",
        )
        st.set_x(0.0)

        composed_path = _ensure_output_path(output_path)
        plt.savefig(composed_path, bbox_inches="tight", pad_inches=0.1)
        plt.close(fig)

        if notebook_mode:
            try:
                img_final = mpimg.imread(composed_path)
                plt.figure(figsize=(12, 12))
                plt.imshow(img_final)
                plt.axis("off")
                plt.show()
            except Exception:
                pass

        try:
            import shutil

            shutil.rmtree(temp_dir)
        except Exception as e:
            print(f"Warning: failed to clean up temp_dir {temp_dir}: {e}")

        print(f"Composed summary image written to {composed_path}")
        return composed_path


In [2]:
bounds_map = {'original': {'bounds': [[0, 1, 2, 3, 0, 1],
   [1, 2, 0, 1, 0, 1],
   [1, 2, 1, 2, 0, 1],
   [1, 2, 2, 3, 0, 1],
   [2, 3, 1, 2, 0, 1]],
  'rotation': 'right:90,rotate:ccw:10,up:10,down:30,down:10,rotate:ccw:10,rotate:ccw:10,down:10'}}

In [5]:
!pip install trame-vtk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 15.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
# !pip install trame-widgets

ERROR: Could not find a version that satisfies the requirement trame-widgets (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for trame-widgets


In [8]:
# !pip install trame-vuetify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 25.3 MB/s eta 0:00:0031m25.6 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
bounds_map

{'original': {'bounds': [[0, 1, 2, 3, 0, 1],
   [1, 2, 0, 1, 0, 1],
   [1, 2, 1, 2, 0, 1],
   [1, 2, 2, 3, 0, 1],
   [2, 3, 1, 2, 0, 1]],
  'rotation': 'right:90,rotate:ccw:10,up:10,down:30,down:10,rotate:ccw:10,rotate:ccw:10,down:10'}}

In [16]:

imagery = StatefulImageryWithInitialRotationModule_8(bounds_map)
imagery.show("original")

Widget(value='<iframe src="http://localhost:39483/index.html?ui=P_0x751129395c70_4&reconnect=auto" class="pyvi…